# Treinamento no Google Colab (GPU)

Antes de executar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.

Pré-requisito: o arquivo `dataset_marcas.zip` (gerado por `python preparar_dataset.py`) deve estar em `MyDrive/car-detection/dataset_marcas.zip` no seu Google Drive.

Se o Colab desconectar no meio do treino, basta rodar todas as células de novo: os experimentos de Dropout já concluídos ficam salvos em `MyDrive/car-detection/checkpoints` e são pulados.

In [ ]:
# 1. Confirma que a GPU está ativa
!nvidia-smi
import tensorflow as tf
print("GPUs disponíveis:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. Clona o repositório
%cd /content
!rm -rf car-detection
!git clone https://github.com/LucasOSantana/car-detection.git
%cd /content/car-detection

In [ ]:
# 3. Monta o Google Drive e descompacta o dataset no disco local do Colab
#    (ler direto do Drive durante o treino é muito lento)
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/car-detection"
!unzip -q -o "{DRIVE_DIR}/dataset_marcas.zip" -d /content/car-detection
!ls /content/car-detection/dataset/marcas/train

In [ ]:
# 4. Executa os experimentos de Dropout (0.0, 0.2, 0.5)
#    Fase 1: até 10 épocas só na cabeça | Fase 2: até 15 épocas de fine-tuning
#    Outros backbones: --backbone efficientnetv2s | convnext_tiny | mobilenetv2
!python car-detection.py --mode train --epochs 10 --epochs-ft 15 --checkpoint-dir "{DRIVE_DIR}/checkpoints"

In [ ]:
# 5. Salva modelo e resultados no Drive (o disco do Colab é apagado ao fim da sessão)
!mkdir -p "{DRIVE_DIR}/saida"
!cp -r car_brand_model.keras class_names.json resultados_experimentos "{DRIVE_DIR}/saida/"
!ls -R "{DRIVE_DIR}/saida"

In [ ]:
# 6. (Opcional) Testa com suas próprias fotos.
#    Dica: nomeie os arquivos começando pela marca (ex.: Volkswagen_1.jpg) para o acerto ser calculado.
import os, shutil
from google.colab import files
os.makedirs("fotos_proprias", exist_ok=True)
for nome in files.upload():
    shutil.move(nome, os.path.join("fotos_proprias", nome))
!python car-detection.py --pasta fotos_proprias
!cp resultados_experimentos/fotos_proprias.csv "{DRIVE_DIR}/saida/"